# Step 3 — Clean FD001 RUL Baseline Model Development

Objectives:

- reload the frozen preprocessing artifacts from Step 2
- define a leakage-safe validation protocol
- establish simple and nonlinear clean-data RUL baselines
- compare raw and capped RUL target formulations
- select modeling choices using validation data only
- keep the official FD001 test set untouched until final evaluation

Primary model inputs:
17 retained and standardized operating/sensor features.

Primary model does not use:
- unit_id
- cycle

Cycle may later be evaluated as an ablation feature.

In [ ]:
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd

# Locate project root
cwd = Path.cwd()

if (cwd / "02_dataset").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "02_dataset").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not locate the project root.")

PROCESSED_DIR = (
    PROJECT_ROOT
    / "02_dataset"
    / "processed"
    / "FD001"
)

print("Project root:", PROJECT_ROOT)
print("Processed directory:", PROCESSED_DIR)
print("Processed directory exists:", PROCESSED_DIR.exists())

In [ ]:
dev_df = pd.read_csv(
    PROCESSED_DIR / "fd001_dev_train_scaled.csv"
)

val_df = pd.read_csv(
    PROCESSED_DIR / "fd001_validation_scaled.csv"
)

scaler = joblib.load(
    PROCESSED_DIR / "fd001_standard_scaler.joblib"
)

with open(
    PROCESSED_DIR / "fd001_preprocessing_config.json",
    "r"
) as f:
    config = json.load(f)

PRIMARY_INPUT_COLUMNS = config["primary_input_columns"]

TARGET_COLUMNS = [
    "RUL_raw",
    "RUL_cap_125",
    "RUL_cap_130"
]

print("Development shape:", dev_df.shape)
print("Validation shape :", val_df.shape)
print("Primary inputs   :", len(PRIMARY_INPUT_COLUMNS))
print("Targets          :", TARGET_COLUMNS)

In [ ]:
assert dev_df.shape == (16561, 22)
assert val_df.shape == (4070, 22)

assert len(PRIMARY_INPUT_COLUMNS) == 17

assert all(
    col in dev_df.columns
    for col in PRIMARY_INPUT_COLUMNS
)

assert all(
    col in val_df.columns
    for col in PRIMARY_INPUT_COLUMNS
)

assert all(
    target in dev_df.columns
    for target in TARGET_COLUMNS
)

assert all(
    target in val_df.columns
    for target in TARGET_COLUMNS
)

print("Step 3 artifacts loaded correctly.")

In [ ]:
EVAL_FRACTIONS = [0.50, 0.60, 0.70, 0.80, 0.90]

snapshot_rows = []

for unit_id, group in val_df.groupby("unit_id"):
    group = group.sort_values("cycle")

    max_cycle = int(group["cycle"].max())

    for fraction in EVAL_FRACTIONS:

        target_cycle = int(
            np.floor(fraction * max_cycle)
        )

        target_cycle = max(1, target_cycle)

        row = group[
            group["cycle"] == target_cycle
        ].copy()

        row["eval_fraction"] = fraction

        snapshot_rows.append(row)


val_snapshots = pd.concat(
    snapshot_rows,
    ignore_index=True
)

print("Validation snapshot rows:",
      len(val_snapshots))

print("Validation engines:",
      val_snapshots["unit_id"].nunique())

print(
    "Snapshots per engine:"
)

print(
    val_snapshots
    .groupby("unit_id")
    .size()
    .value_counts()
)

In [ ]:
assert len(val_snapshots) == 100

assert (
    val_snapshots["unit_id"].nunique()
    == 20
)

assert (
    val_snapshots
    .groupby("unit_id")
    .size()
    .eq(5)
    .all()
)

assert val_snapshots.isna().sum().sum() == 0

print("Balanced validation snapshot checks passed.")

In [ ]:
display(
    val_snapshots[
        [
            "unit_id",
            "cycle",
            "RUL_raw",
            "RUL_cap_125",
            "RUL_cap_130",
            "eval_fraction"
        ]
    ].head(15)
)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)


def regression_metrics(y_true, y_pred):

    return {
        "RMSE": root_mean_squared_error(
            y_true,
            y_pred
        ),

        "MAE": mean_absolute_error(
            y_true,
            y_pred
        ),

        "R2": r2_score(
            y_true,
            y_pred
        )
    }


print("Evaluation metric function ready.")

In [ ]:
from sklearn.dummy import DummyRegressor

X_dev = dev_df[PRIMARY_INPUT_COLUMNS]

X_val_snapshots = val_snapshots[
    PRIMARY_INPUT_COLUMNS
]

X_val_full = val_df[
    PRIMARY_INPUT_COLUMNS
]

baseline_results = []

In [ ]:
for target in TARGET_COLUMNS:

    y_dev = dev_df[target]

    y_snapshot = val_snapshots[target]
    y_full = val_df[target]

    model = DummyRegressor(
        strategy="mean"
    )

    model.fit(
        X_dev,
        y_dev
    )

    snapshot_pred = model.predict(
        X_val_snapshots
    )

    full_pred = model.predict(
        X_val_full
    )

    snapshot_metrics = regression_metrics(
        y_snapshot,
        snapshot_pred
    )

    full_metrics = regression_metrics(
        y_full,
        full_pred
    )

    baseline_results.append({
        "model": "DummyMean",
        "target": target,
        "evaluation": "balanced_snapshots",
        **snapshot_metrics
    })

    baseline_results.append({
        "model": "DummyMean",
        "target": target,
        "evaluation": "full_validation",
        **full_metrics
    })

In [ ]:
baseline_results_df = pd.DataFrame(
    baseline_results
)

display(
    baseline_results_df.round(3)
)

In [ ]:
for target in TARGET_COLUMNS:

    model = DummyRegressor(
        strategy="mean"
    )

    model.fit(
        X_dev,
        dev_df[target]
    )

    print(
        f"{target:12s} -> "
        f"constant prediction = "
        f"{model.constant_[0][0]:.3f}"
    )

In [ ]:
from sklearn.linear_model import LinearRegression

linear_results = []
linear_models = {}

for target in TARGET_COLUMNS:

    y_dev = dev_df[target]
    y_snapshot = val_snapshots[target]
    y_full = val_df[target]

    model = LinearRegression()

    model.fit(
        X_dev,
        y_dev
    )

    linear_models[target] = model

    snapshot_pred = model.predict(
        X_val_snapshots
    )

    full_pred = model.predict(
        X_val_full
    )

    snapshot_metrics = regression_metrics(
        y_snapshot,
        snapshot_pred
    )

    full_metrics = regression_metrics(
        y_full,
        full_pred
    )

    linear_results.append({
        "model": "LinearRegression",
        "target": target,
        "evaluation": "balanced_snapshots",
        **snapshot_metrics
    })

    linear_results.append({
        "model": "LinearRegression",
        "target": target,
        "evaluation": "full_validation",
        **full_metrics
    })

In [ ]:
linear_results_df = pd.DataFrame(
    linear_results
)

display(
    linear_results_df.round(3)
)

In [ ]:
baseline_comparison_df = pd.concat(
    [
        baseline_results_df,
        linear_results_df
    ],
    ignore_index=True
)

display(
    baseline_comparison_df.round(3)
)

In [ ]:
comparison_rows = []

for target in TARGET_COLUMNS:

    dummy_rmse = baseline_results_df[
        (baseline_results_df["target"] == target) &
        (baseline_results_df["evaluation"] == "balanced_snapshots")
    ]["RMSE"].iloc[0]

    linear_rmse = linear_results_df[
        (linear_results_df["target"] == target) &
        (linear_results_df["evaluation"] == "balanced_snapshots")
    ]["RMSE"].iloc[0]

    improvement_pct = (
        (dummy_rmse - linear_rmse)
        / dummy_rmse
        * 100
    )

    comparison_rows.append({
        "target": target,
        "dummy_RMSE": dummy_rmse,
        "linear_RMSE": linear_rmse,
        "RMSE_improvement_%": improvement_pct
    })


linear_improvement_df = pd.DataFrame(
    comparison_rows
)

display(
    linear_improvement_df.round(3)
)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_results = []
rf_models = {}

for target in TARGET_COLUMNS:

    y_dev = dev_df[target]
    y_snapshot = val_snapshots[target]
    y_full = val_df[target]

    model = RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_dev,
        y_dev
    )

    rf_models[target] = model

    snapshot_pred = model.predict(
        X_val_snapshots
    )

    full_pred = model.predict(
        X_val_full
    )

    snapshot_metrics = regression_metrics(
        y_snapshot,
        snapshot_pred
    )

    full_metrics = regression_metrics(
        y_full,
        full_pred
    )

    rf_results.append({
        "model": "RandomForest",
        "target": target,
        "evaluation": "balanced_snapshots",
        **snapshot_metrics
    })

    rf_results.append({
        "model": "RandomForest",
        "target": target,
        "evaluation": "full_validation",
        **full_metrics
    })

In [ ]:
rf_results_df = pd.DataFrame(
    rf_results
)

display(
    rf_results_df.round(3)
)

In [ ]:
clean_baseline_results_df = pd.concat(
    [
        baseline_results_df,
        linear_results_df,
        rf_results_df
    ],
    ignore_index=True
)

display(
    clean_baseline_results_df.round(3)
)

In [ ]:
snapshot_comparison = (
    clean_baseline_results_df[
        clean_baseline_results_df["evaluation"]
        == "balanced_snapshots"
    ]
    .pivot(
        index="target",
        columns="model",
        values="RMSE"
    )
)

display(
    snapshot_comparison.round(3)
)

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

hgb_results = []
hgb_models = {}

for target in TARGET_COLUMNS:

    y_dev = dev_df[target]
    y_snapshot = val_snapshots[target]
    y_full = val_df[target]

    model = HistGradientBoostingRegressor(
        random_state=42,
        early_stopping=False
    )

    model.fit(
        X_dev,
        y_dev
    )

    hgb_models[target] = model

    snapshot_pred = model.predict(
        X_val_snapshots
    )

    full_pred = model.predict(
        X_val_full
    )

    snapshot_metrics = regression_metrics(
        y_snapshot,
        snapshot_pred
    )

    full_metrics = regression_metrics(
        y_full,
        full_pred
    )

    hgb_results.append({
        "model": "HistGradientBoosting",
        "target": target,
        "evaluation": "balanced_snapshots",
        **snapshot_metrics
    })

    hgb_results.append({
        "model": "HistGradientBoosting",
        "target": target,
        "evaluation": "full_validation",
        **full_metrics
    })

In [ ]:
hgb_results_df = pd.DataFrame(
    hgb_results
)

display(
    hgb_results_df.round(3)
)

In [ ]:
all_clean_results_df = pd.concat(
    [
        baseline_results_df,
        linear_results_df,
        rf_results_df,
        hgb_results_df
    ],
    ignore_index=True
)

snapshot_comparison = (
    all_clean_results_df[
        all_clean_results_df["evaluation"]
        == "balanced_snapshots"
    ]
    .pivot(
        index="target",
        columns="model",
        values="RMSE"
    )
)

display(
    snapshot_comparison.round(3)
)

In [ ]:
def create_sequences(
    df,
    feature_columns,
    target_column,
    window_size
):
    X_sequences = []
    y_sequences = []

    metadata = []

    for unit_id, group in df.groupby("unit_id"):

        group = (
            group
            .sort_values("cycle")
            .reset_index(drop=True)
        )

        features = group[
            feature_columns
        ].to_numpy(dtype=np.float32)

        targets = group[
            target_column
        ].to_numpy(dtype=np.float32)

        cycles = group[
            "cycle"
        ].to_numpy()

        if len(group) < window_size:
            continue

        for end_idx in range(
            window_size - 1,
            len(group)
        ):

            start_idx = (
                end_idx - window_size + 1
            )

            X_sequences.append(
                features[
                    start_idx:end_idx + 1
                ]
            )

            y_sequences.append(
                targets[end_idx]
            )

            metadata.append({
                "unit_id": unit_id,
                "cycle": cycles[end_idx]
            })

    return (
        np.asarray(
            X_sequences,
            dtype=np.float32
        ),
        np.asarray(
            y_sequences,
            dtype=np.float32
        ),
        pd.DataFrame(metadata)
    )

In [ ]:
WINDOW_SIZE = 30

In [ ]:
WINDOW_SIZE = 30

X_dev_seq, y_dev_seq, dev_seq_meta = (
    create_sequences(
        dev_df,
        PRIMARY_INPUT_COLUMNS,
        "RUL_cap_125",
        WINDOW_SIZE
    )
)

X_val_seq, y_val_seq, val_seq_meta = (
    create_sequences(
        val_df,
        PRIMARY_INPUT_COLUMNS,
        "RUL_cap_125",
        WINDOW_SIZE
    )
)

print("Development sequences:", X_dev_seq.shape)
print("Development targets  :", y_dev_seq.shape)

print("\nValidation sequences :", X_val_seq.shape)
print("Validation targets   :", y_val_seq.shape)

In [ ]:
    assert X_dev_seq.ndim == 3
assert X_val_seq.ndim == 3

assert X_dev_seq.shape[1] == WINDOW_SIZE
assert X_val_seq.shape[1] == WINDOW_SIZE

assert X_dev_seq.shape[2] == 17
assert X_val_seq.shape[2] == 17

assert len(X_dev_seq) == len(y_dev_seq)
assert len(X_val_seq) == len(y_val_seq)

assert len(dev_seq_meta) == len(y_dev_seq)
assert len(val_seq_meta) == len(y_val_seq)

assert not np.isnan(X_dev_seq).any()
assert not np.isnan(X_val_seq).any()

assert not np.isinf(X_dev_seq).any()
assert not np.isinf(X_val_seq).any()

print("Sequence integrity checks passed.")

In [ ]:
expected_dev_sequences = sum(
    max(0, len(group) - WINDOW_SIZE + 1)
    for _, group in dev_df.groupby("unit_id")
)

expected_val_sequences = sum(
    max(0, len(group) - WINDOW_SIZE + 1)
    for _, group in val_df.groupby("unit_id")
)

print("Development sequence shape:", X_dev_seq.shape)
print("Expected development sequences:", expected_dev_sequences)

print("\nValidation sequence shape:", X_val_seq.shape)
print("Expected validation sequences:", expected_val_sequences)

assert len(X_dev_seq) == expected_dev_sequences
assert len(X_val_seq) == expected_val_sequences

print("\nSequence-count checks passed.")

In [ ]:
# Give every validation sequence an index
val_seq_lookup = (
    val_seq_meta
    .reset_index()
    .rename(columns={"index": "sequence_index"})
)

# The exact 100 validation checkpoints already used
snapshot_keys = val_snapshots[
    [
        "unit_id",
        "cycle",
        "RUL_cap_125",
        "eval_fraction"
    ]
].copy()

# Find the corresponding sequence for every checkpoint
val_seq_snapshot_map = snapshot_keys.merge(
    val_seq_lookup,
    on=["unit_id", "cycle"],
    how="left",
    validate="one_to_one"
)

print("Matched sequence snapshots:",
      len(val_seq_snapshot_map))

print(
    "Snapshots without a matching sequence:",
    val_seq_snapshot_map["sequence_index"].isna().sum()
)

In [ ]:
snapshot_seq_indices = (
    val_seq_snapshot_map["sequence_index"]
    .astype(int)
    .to_numpy()
)

X_val_seq_snapshots = X_val_seq[
    snapshot_seq_indices
]

y_val_seq_snapshots = y_val_seq[
    snapshot_seq_indices
]

assert len(X_val_seq_snapshots) == 100
assert len(y_val_seq_snapshots) == 100

assert np.allclose(
    y_val_seq_snapshots,
    val_seq_snapshot_map["RUL_cap_125"].to_numpy()
)

print("Balanced sequence-validation set ready.")
print("Shape:", X_val_seq_snapshots.shape)

In [ ]:
try:
    import torch

    print("PyTorch version:", torch.__version__)
    print("CUDA available :", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("GPU acceleration is not currently available.")

except ModuleNotFoundError:
    print("PyTorch is not installed yet.")

In [ ]:
import torch

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())
print("PyTorch CUDA    :", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))
    print(
        "GPU memory (GB) :",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024 ** 3),
            2
        )
    )

In [ ]:
required_objects = [
    "X_dev_seq",
    "y_dev_seq",
    "X_val_seq",
    "y_val_seq",
    "X_val_seq_snapshots",
    "y_val_seq_snapshots",
    "PRIMARY_INPUT_COLUMNS",
]

for name in required_objects:
    print(name, ":", name in globals())

In [ ]:
import random
import torch
import torch.nn as nn

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from torch.utils.data import (
    TensorDataset,
    DataLoader
)

X_dev_tensor = torch.from_numpy(
    X_dev_seq
).float()

y_dev_tensor = torch.from_numpy(
    y_dev_seq
).float().unsqueeze(1)

X_val_tensor = torch.from_numpy(
    X_val_seq
).float()

y_val_tensor = torch.from_numpy(
    y_val_seq
).float().unsqueeze(1)

X_snapshot_tensor = torch.from_numpy(
    X_val_seq_snapshots
).float()

y_snapshot_tensor = torch.from_numpy(
    y_val_seq_snapshots
).float().unsqueeze(1)


train_dataset = TensorDataset(
    X_dev_tensor,
    y_dev_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

snapshot_dataset = TensorDataset(
    X_snapshot_tensor,
    y_snapshot_tensor
)

print("Training samples :", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Snapshot samples  :", len(snapshot_dataset))

In [ ]:
BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

snapshot_loader = DataLoader(
    snapshot_dataset,
    batch_size=100,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("DataLoaders ready.")

In [ ]:
class GRURegressor(nn.Module):

    def __init__(
        self,
        input_size,
        hidden_size=64,
        num_layers=1,
        dropout=0.20
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        output, hidden = self.gru(x)

        last_hidden = hidden[-1]

        prediction = self.regressor(last_hidden)

        return prediction

In [ ]:
gru_model = GRURegressor(
    input_size=len(PRIMARY_INPUT_COLUMNS),
    hidden_size=64,
    num_layers=1,
    dropout=0.20
).to(DEVICE)

print(gru_model)

In [ ]:
total_params = sum(
    p.numel()
    for p in gru_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in gru_model.parameters()
    if p.requires_grad
)

print("Total parameters    :", total_params)
print("Trainable parameters:", trainable_params)

In [ ]:
print("Device:", DEVICE)
print("Training samples :", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Snapshot samples  :", len(snapshot_dataset))

In [ ]:
import copy
import math

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    gru_model.parameters(),
    lr=1e-3
)

MAX_EPOCHS = 100
PATIENCE = 12

best_val_rmse = float("inf")
best_epoch = 0
best_model_state = None

epochs_without_improvement = 0

training_history = []

In [ ]:
for epoch in range(1, MAX_EPOCHS + 1):

    # -------------------------
    # Training
    # -------------------------
    gru_model.train()

    train_squared_error = 0.0
    train_count = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(
            DEVICE,
            non_blocking=True
        )

        y_batch = y_batch.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad()

        predictions = gru_model(X_batch)

        loss = criterion(
            predictions,
            y_batch
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            gru_model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        train_squared_error += (
            loss.item() * len(X_batch)
        )

        train_count += len(X_batch)

    train_rmse = math.sqrt(
        train_squared_error / train_count
    )


    # -------------------------
    # Validation
    # -------------------------
    gru_model.eval()

    val_squared_error = 0.0
    val_count = 0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(
                DEVICE,
                non_blocking=True
            )

            y_batch = y_batch.to(
                DEVICE,
                non_blocking=True
            )

            predictions = gru_model(X_batch)

            loss = criterion(
                predictions,
                y_batch
            )

            val_squared_error += (
                loss.item() * len(X_batch)
            )

            val_count += len(X_batch)

    val_rmse = math.sqrt(
        val_squared_error / val_count
    )


    training_history.append({
        "epoch": epoch,
        "train_RMSE": train_rmse,
        "val_RMSE": val_rmse
    })


    # -------------------------
    # Early stopping
    # -------------------------
    if val_rmse < best_val_rmse:

        best_val_rmse = val_rmse
        best_epoch = epoch

        best_model_state = copy.deepcopy(
            gru_model.state_dict()
        )

        epochs_without_improvement = 0

    else:
        epochs_without_improvement += 1


    print(
        f"Epoch {epoch:03d} | "
        f"Train RMSE: {train_rmse:7.3f} | "
        f"Val RMSE: {val_rmse:7.3f} | "
        f"Best: {best_val_rmse:7.3f}"
    )


    if epochs_without_improvement >= PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch}."
        )

        break

In [ ]:
gru_model.load_state_dict(
    best_model_state
)

print("Best epoch   :", best_epoch)
print("Best val RMSE:", round(best_val_rmse, 3))

In [ ]:
gru_model.eval()

snapshot_predictions = []
snapshot_targets = []

with torch.no_grad():

    for X_batch, y_batch in snapshot_loader:

        X_batch = X_batch.to(
            DEVICE,
            non_blocking=True
        )

        predictions = gru_model(X_batch)

        snapshot_predictions.extend(
            predictions
            .cpu()
            .numpy()
            .ravel()
        )

        snapshot_targets.extend(
            y_batch
            .numpy()
            .ravel()
        )


snapshot_predictions = np.asarray(
    snapshot_predictions
)

snapshot_targets = np.asarray(
    snapshot_targets
)

print("Predictions:", snapshot_predictions.shape)
print("Targets    :", snapshot_targets.shape)

In [ ]:
gru_snapshot_metrics = regression_metrics(
    snapshot_targets,
    snapshot_predictions
)

print("GRU balanced-snapshot metrics:")

for metric, value in gru_snapshot_metrics.items():
    print(f"{metric}: {value:.3f}")

In [ ]:
gru_result_row = pd.DataFrame([
    {
        "model": "GRU",
        "target": "RUL_cap_125",
        "evaluation": "balanced_snapshots",
        **gru_snapshot_metrics
    }
])

cap125_snapshot_comparison = pd.concat(
    [
        all_clean_results_df[
            (all_clean_results_df["target"] == "RUL_cap_125")
            &
            (
                all_clean_results_df["evaluation"]
                == "balanced_snapshots"
            )
        ],
        gru_result_row
    ],
    ignore_index=True
)

display(
    cap125_snapshot_comparison[
        ["model", "RMSE", "MAE", "R2"]
    ]
    .sort_values("RMSE")
    .round(3)
)

In [ ]:
print(
    "Minimum GRU prediction:",
    round(snapshot_predictions.min(), 3)
)

print(
    "Maximum GRU prediction:",
    round(snapshot_predictions.max(), 3)
)

print(
    "Negative predictions:",
    (snapshot_predictions < 0).sum()
)

In [ ]:
MODELS_DIR = (
    PROJECT_ROOT
    / "05_experiments"
    / "models"
)

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

gru_model_path = (
    MODELS_DIR
    / "fd001_gru_cap125_window30_seed42.pt"
)

torch.save(
    {
        "model_state_dict": gru_model.state_dict(),
        "best_epoch": best_epoch,
        "best_val_rmse": best_val_rmse,
        "window_size": WINDOW_SIZE,
        "input_columns": PRIMARY_INPUT_COLUMNS,
        "target": "RUL_cap_125",
        "seed": SEED
    },
    gru_model_path
)

print("GRU model saved to:")
print(gru_model_path)

In [ ]:
gru_history_df = pd.DataFrame(
    training_history
)

history_path = (
    PROJECT_ROOT
    / "05_experiments"
    / "gru_cap125_training_history.csv"
)

gru_history_df.to_csv(
    history_path,
    index=False
)

print("Training history saved to:")
print(history_path)

In [ ]:
RESULTS_DIR = (
    PROJECT_ROOT
    / "06_results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

gru_snapshot_result = pd.DataFrame([
    {
        "model": "GRU",
        "target": "RUL_cap_125",
        "window_size": WINDOW_SIZE,
        "best_epoch": best_epoch,
        "full_val_best_RMSE": best_val_rmse,
        "snapshot_RMSE": gru_snapshot_metrics["RMSE"],
        "snapshot_MAE": gru_snapshot_metrics["MAE"],
        "snapshot_R2": gru_snapshot_metrics["R2"]
    }
])

gru_snapshot_result.to_csv(
    RESULTS_DIR / "fd001_gru_cap125_baseline_result.csv",
    index=False
)

print("GRU result saved.")

In [ ]:
def train_gru_for_target(
    target_column,
    window_size=30,
    seed=42,
    max_epochs=100,
    patience=12
):
    import copy
    import math
    import random

    # Reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Build sequences for this target
    X_dev_t, y_dev_t, dev_meta_t = create_sequences(
        dev_df,
        PRIMARY_INPUT_COLUMNS,
        target_column,
        window_size
    )

    X_val_t, y_val_t, val_meta_t = create_sequences(
        val_df,
        PRIMARY_INPUT_COLUMNS,
        target_column,
        window_size
    )

    # Convert to tensors
    train_dataset_t = TensorDataset(
        torch.from_numpy(X_dev_t).float(),
        torch.from_numpy(y_dev_t).float().unsqueeze(1)
    )

    val_dataset_t = TensorDataset(
        torch.from_numpy(X_val_t).float(),
        torch.from_numpy(y_val_t).float().unsqueeze(1)
    )

    train_loader_t = DataLoader(
        train_dataset_t,
        batch_size=128,
        shuffle=True,
        num_workers=0,
        pin_memory=True
    )

    val_loader_t = DataLoader(
        val_dataset_t,
        batch_size=256,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    # Fresh model for each target
    model = GRURegressor(
        input_size=len(PRIMARY_INPUT_COLUMNS),
        hidden_size=64,
        num_layers=1,
        dropout=0.20
    ).to(DEVICE)

    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-3
    )

    best_val_rmse = float("inf")
    best_epoch = 0
    best_state = None
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, max_epochs + 1):

        # Training
        model.train()

        train_se = 0.0
        train_count = 0

        for X_batch, y_batch in train_loader_t:

            X_batch = X_batch.to(
                DEVICE,
                non_blocking=True
            )

            y_batch = y_batch.to(
                DEVICE,
                non_blocking=True
            )

            optimizer.zero_grad()

            pred = model(X_batch)

            loss = criterion(pred, y_batch)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            train_se += (
                loss.item() * len(X_batch)
            )

            train_count += len(X_batch)

        train_rmse = math.sqrt(
            train_se / train_count
        )

        # Validation
        model.eval()

        val_se = 0.0
        val_count = 0

        with torch.no_grad():

            for X_batch, y_batch in val_loader_t:

                X_batch = X_batch.to(
                    DEVICE,
                    non_blocking=True
                )

                y_batch = y_batch.to(
                    DEVICE,
                    non_blocking=True
                )

                pred = model(X_batch)

                loss = criterion(pred, y_batch)

                val_se += (
                    loss.item() * len(X_batch)
                )

                val_count += len(X_batch)

        val_rmse = math.sqrt(
            val_se / val_count
        )

        history.append({
            "epoch": epoch,
            "train_RMSE": train_rmse,
            "val_RMSE": val_rmse
        })

        if val_rmse < best_val_rmse:

            best_val_rmse = val_rmse
            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            epochs_without_improvement = 0

        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    model.load_state_dict(best_state)

    return {
        "model": model,
        "best_epoch": best_epoch,
        "best_val_rmse": best_val_rmse,
        "history": pd.DataFrame(history),
        "val_meta": val_meta_t,
        "X_val": X_val_t,
        "y_val": y_val_t
    }

In [ ]:
print("Training GRU for RUL_raw...")

gru_raw_result = train_gru_for_target(
    target_column="RUL_raw",
    window_size=WINDOW_SIZE,
    seed=SEED,
    max_epochs=100,
    patience=12
)

print(
    "RUL_raw best epoch:",
    gru_raw_result["best_epoch"]
)

print(
    "RUL_raw best full-validation RMSE:",
    round(
        gru_raw_result["best_val_rmse"],
        3
    )
)

In [ ]:
print("Training GRU for RUL_cap_130...")

gru_cap130_result = train_gru_for_target(
    target_column="RUL_cap_130",
    window_size=WINDOW_SIZE,
    seed=SEED,
    max_epochs=100,
    patience=12
)

print(
    "RUL_cap_130 best epoch:",
    gru_cap130_result["best_epoch"]
)

print(
    "RUL_cap_130 best full-validation RMSE:",
    round(
        gru_cap130_result["best_val_rmse"],
        3
    )
)

In [ ]:
print("Training controlled GRU for RUL_cap_125...")

gru_cap125_controlled = train_gru_for_target(
    target_column="RUL_cap_125",
    window_size=WINDOW_SIZE,
    seed=SEED,
    max_epochs=100,
    patience=12
)

print(
    "RUL_cap_125 best epoch:",
    gru_cap125_controlled["best_epoch"]
)

print(
    "RUL_cap_125 best full-validation RMSE:",
    round(
        gru_cap125_controlled["best_val_rmse"],
        3
    )
)

In [ ]:
def evaluate_gru_target_on_snapshots(
    result,
    trained_target
):
    # Match sequence endpoints with the exact
    # same 100 balanced validation checkpoints
    seq_lookup = (
        result["val_meta"]
        .reset_index()
        .rename(columns={"index": "sequence_index"})
    )

    snapshot_info = val_snapshots[
        [
            "unit_id",
            "cycle",
            "RUL_raw",
            "RUL_cap_125",
            "RUL_cap_130"
        ]
    ].copy()

    matched = snapshot_info.merge(
        seq_lookup,
        on=["unit_id", "cycle"],
        how="left",
        validate="one_to_one"
    )

    assert matched["sequence_index"].isna().sum() == 0
    assert len(matched) == 100

    indices = (
        matched["sequence_index"]
        .astype(int)
        .to_numpy()
    )

    X_snapshot = result["X_val"][indices]

    model = result["model"]
    model.eval()

    with torch.no_grad():

        X_tensor = torch.from_numpy(
            X_snapshot
        ).float().to(DEVICE)

        predictions = (
            model(X_tensor)
            .cpu()
            .numpy()
            .ravel()
        )

    # Metric against the target used for training
    own_target_metrics = regression_metrics(
        matched[trained_target].to_numpy(),
        predictions
    )

    # Common physical reference:
    # evaluate every model against raw RUL too
    raw_reference_metrics = regression_metrics(
        matched["RUL_raw"].to_numpy(),
        predictions
    )

    return {
        "target": trained_target,

        "best_epoch":
            result["best_epoch"],

        "full_val_best_RMSE":
            result["best_val_rmse"],

        "own_target_snapshot_RMSE":
            own_target_metrics["RMSE"],

        "own_target_snapshot_MAE":
            own_target_metrics["MAE"],

        "own_target_snapshot_R2":
            own_target_metrics["R2"],

        "raw_reference_RMSE":
            raw_reference_metrics["RMSE"],

        "raw_reference_MAE":
            raw_reference_metrics["MAE"],

        "raw_reference_R2":
            raw_reference_metrics["R2"]
    }

In [ ]:
gru_target_comparison = pd.DataFrame([
    evaluate_gru_target_on_snapshots(
        gru_raw_result,
        "RUL_raw"
    ),

    evaluate_gru_target_on_snapshots(
        gru_cap125_controlled,
        "RUL_cap_125"
    ),

    evaluate_gru_target_on_snapshots(
        gru_cap130_result,
        "RUL_cap_130"
    )
])

display(
    gru_target_comparison.round(3)
)

In [ ]:
gru_target_comparison.to_csv(
    RESULTS_DIR / "fd001_gru_target_comparison.csv",
    index=False
)

print("GRU target comparison saved.")

In [ ]:
selected_gru_path = (
    MODELS_DIR
    / "fd001_gru_selected_cap125_window30_seed42.pt"
)

torch.save(
    {
        "model_state_dict":
            gru_cap125_controlled["model"].state_dict(),

        "best_epoch":
            gru_cap125_controlled["best_epoch"],

        "best_val_rmse":
            gru_cap125_controlled["best_val_rmse"],

        "window_size":
            WINDOW_SIZE,

        "input_columns":
            PRIMARY_INPUT_COLUMNS,

        "target":
            "RUL_cap_125",

        "seed":
            SEED
    },
    selected_gru_path
)

print("Selected GRU saved to:")
print(selected_gru_path)

In [ ]:
selected_gru = gru_cap125_controlled["model"]
selected_gru.eval()

X_val_all = gru_cap125_controlled["X_val"]
val_meta_all = gru_cap125_controlled["val_meta"].copy()

with torch.no_grad():

    X_val_all_tensor = torch.from_numpy(
        X_val_all
    ).float().to(DEVICE)

    val_predictions = (
        selected_gru(X_val_all_tensor)
        .cpu()
        .numpy()
        .ravel()
    )

print("Validation predictions:", val_predictions.shape)

In [ ]:
val_error_df = val_meta_all.copy()

val_error_df = val_error_df.merge(
    val_df[
        [
            "unit_id",
            "cycle",
            "RUL_raw",
            "RUL_cap_125"
        ]
    ],
    on=["unit_id", "cycle"],
    how="left",
    validate="one_to_one"
)

val_error_df["prediction"] = val_predictions

val_error_df["error"] = (
    val_error_df["prediction"]
    - val_error_df["RUL_raw"]
)

val_error_df["absolute_error"] = (
    val_error_df["error"].abs()
)

print("Rows:", len(val_error_df))
print("Missing values:", val_error_df.isna().sum().sum())

In [ ]:
bins = [
    -0.1,
    30,
    60,
    90,
    125,
    np.inf
]

labels = [
    "0-30",
    "31-60",
    "61-90",
    "91-125",
    ">125"
]

val_error_df["RUL_region"] = pd.cut(
    val_error_df["RUL_raw"],
    bins=bins,
    labels=labels
)

In [ ]:
region_results = []

for region in labels:

    region_df = val_error_df[
        val_error_df["RUL_region"] == region
    ]

    if len(region_df) == 0:
        continue

    metrics = regression_metrics(
        region_df["RUL_raw"],
        region_df["prediction"]
    )

    region_results.append({
        "RUL_region": region,
        "samples": len(region_df),
        "RMSE": metrics["RMSE"],
        "MAE": metrics["MAE"],
        "mean_error": region_df["error"].mean()
    })


region_results_df = pd.DataFrame(
    region_results
)

display(
    region_results_df.round(3)
)

In [ ]:
near_eol = val_error_df[
    val_error_df["RUL_raw"] <= 30
].copy()

print("Near-EOL samples:", len(near_eol))

print(
    "Mean prediction error:",
    round(near_eol["error"].mean(), 3)
)

print(
    "Overestimations > 10 cycles:",
    (near_eol["error"] > 10).sum()
)

print(
    "Percentage overestimated by >10 cycles:",
    round(
        100 * (near_eol["error"] > 10).mean(),
        2
    ),
    "%"
)

In [ ]:
region_results_df.to_csv(
    RESULTS_DIR / "fd001_gru_cap125_rul_region_diagnostics.csv",
    index=False
)

near_eol_summary = pd.DataFrame([{
    "samples": len(near_eol),
    "RMSE": regression_metrics(
        near_eol["RUL_raw"],
        near_eol["prediction"]
    )["RMSE"],
    "MAE": regression_metrics(
        near_eol["RUL_raw"],
        near_eol["prediction"]
    )["MAE"],
    "mean_error": near_eol["error"].mean(),
    "overestimation_gt_10_count":
        int((near_eol["error"] > 10).sum()),
    "overestimation_gt_10_pct":
        100 * (near_eol["error"] > 10).mean()
}])

near_eol_summary.to_csv(
    RESULTS_DIR / "fd001_gru_cap125_near_eol_diagnostics.csv",
    index=False
)

print("Step 3 diagnostic results saved.")